# 5. Annotating with LLMs

This notebook uses a stand-in model that costs nothing. To use a real model replace `standin_model()` with `ModelHub.from_env()["chatgpt"]` (needs a key, see `docs/reference.md`).

In [1]:
import tempfile
from pathlib import Path

import pandas as pd

from polalign import run_dataset
from polalign.toy import toy_ratings, toy_spec

pd.set_option("display.width", 160)
result = run_dataset(toy_spec(), raw=toy_ratings())        # the toy dataset: 60 texts, 24 annotators, gender / politics / age

In [2]:
from polalign import annotate, load_model_inputs, save_au_tables
from polalign.toy import standin_model

out = Path(tempfile.mkdtemp())
save_au_tables({"toy": result.au}, out)                                   # aposteriori_for_models.csv: the attributed texts
inputs = load_model_inputs(out / "aposteriori_for_models.csv")
{name: len(table) for name, table in inputs.items()}

{'toy': 40}

## Your prompt

You write a function. `texts` is the batch already numbered, `item` the word for a text.

In [3]:
def my_prompt(texts, item):
    return f"""Rate how toxic each {item} is on a scale from 1 to 5.
Reply with one line per {item}: score=<1-5>, label=<toxic/civil>
A score of 1 or 2 is civil, 3 or more is toxic.

{texts}"""

model = standin_model()
annotate(model, inputs, out, prompt=my_prompt, run="baseline", item_words={"toy": "comment"}, show_progress=False)

,texts,batches,resumed,model calls,seconds
dataset,,,,,
toy,40,4,0,4,0.0


## What was saved

In [4]:
ratings = pd.read_csv(out / "toy" / "baseline" / "standin.csv")
ratings.head()

,dataset,item_id,text,model,run,score,label
0,toy,t2,Toy comment number 2.,standin,baseline,1,civil
1,toy,t5,Toy comment number 5!,standin,baseline,5,toxic
2,toy,t8,Toy comment number 8.,standin,baseline,1,civil
3,toy,t11,Toy comment number 11!,standin,baseline,5,toxic
4,toy,t14,Toy comment number 14.,standin,baseline,1,civil


In [5]:
from polalign import inspect_run

inspect_run(out, "standin", run="baseline")

,texts,mean score,toxic %,score 1,score 2,score 3,score 4,score 5,batches asked again,batches not parsed
toy,40.0,3.0,50.0,20.0,0.0,0.0,0.0,20.0,0.0,0.0


A second run with another prompt goes in another `run` folder, so the two stay separate. Running the same call again resumes: finished batches are not asked again.